# 18.7 贝叶斯优化 / Bayesian Optimization

**中文**：Part 18 的收官,也是贝叶斯方法最漂亮的落地应用。想象你要优化一个**又贵又黑的函数**:每评估一次都很昂贵(训练一个大模型看验证集精度、跑一次真实实验、做一次 A/B),而且没有梯度、可能有很多局部最优。你**评估次数极其有限**,怎么用最少的评估找到最优?普通网格/随机搜索太浪费。**贝叶斯优化(Bayesian Optimization, BO)** 给出聪明答案:*"用一个 GP 边评估边建立对这个黑箱的'信念',每一步都用不确定性来智能地决定下一个最值得试的点。"* 它是**自动调参(AutoML)、材料/药物发现、实验设计**的核心武器。
**English**: The finale of Part 18, and the most beautiful practical application of Bayesian methods. Imagine optimizing an **expensive black-box function**: each evaluation is costly (train a big model and check validation accuracy, run a real experiment, do an A/B test), with no gradients and possibly many local optima. With **very few evaluations allowed**, how do you find the optimum? Grid/random search is wasteful. **Bayesian Optimization (BO)** gives a clever answer: *"use a GP to build a 'belief' about the black-box as you evaluate, and at each step use uncertainty to intelligently pick the most worthwhile next point."* It is the core weapon of **automated hyperparameter tuning (AutoML), materials/drug discovery, and experiment design**.

---

**中文**：BO 的循环极其简洁,三步不断重复:
**English**: The BO loop is elegantly simple, repeating three steps:
1. **拟合代理模型(surrogate)**:用目前所有 $(x,f(x))$ 评估拟合一个 **GP**(18.6),得到对黑箱的均值 + 不确定性。
   **Fit a surrogate**: fit a **GP** (18.6) on all $(x,f(x))$ so far, giving a mean + uncertainty over the black-box.
2. **最大化采集函数(acquisition)**:用一个**采集函数**衡量"每个候选点有多值得试",取它最大的点作为下一个评估点。
   **Maximize an acquisition function**: an **acquisition function** scores "how worthwhile each candidate is"; pick its maximizer as the next point.
3. **评估 + 更新**:在该点真正评估昂贵函数,把结果加进数据,回到第1步。
   **Evaluate + update**: actually evaluate the expensive function there, add it, and go back to step 1.

**中文**：灵魂在**采集函数**——它必须平衡**探索(exploration,去不确定性大的地方,可能有惊喜)** 和 **利用(exploitation,去预测值高的地方,稳拿收益)**。三个经典采集函数:
**English**: The soul is the **acquisition function** — it must balance **exploration (go where uncertainty is high, maybe a surprise)** and **exploitation (go where the predicted value is high, a safe gain)**. Three classics:
- **期望改进 EI(Expected Improvement)**:$\text{EI}(x)=\mathbb E[\max(f(x)-f^+,0)]$,当前最优 $f^+$ 之上的**期望提升量**。最常用,自动平衡探索利用。
  **Expected Improvement (EI)**: $\text{EI}(x)=\mathbb E[\max(f(x)-f^+,0)]$, the expected improvement over the current best $f^+$. Most popular, auto-balances explore/exploit.
- **置信上界 UCB(Upper Confidence Bound)**:$\text{UCB}(x)=\mu(x)+\kappa\,\sigma(x)$,"预测均值 + $\kappa$ 倍不确定性"。$\kappa$ 直接调节探索强度(和 17.10 的 LinUCB 同源!)。
  **Upper Confidence Bound (UCB)**: $\text{UCB}(x)=\mu(x)+\kappa\,\sigma(x)$, "predicted mean + $\kappa$× uncertainty." $\kappa$ directly tunes exploration (same idea as LinUCB in 17.10!).
- **改进概率 PI(Probability of Improvement)**:只看"超过当前最优的概率",偏保守(易过度利用)。
  **Probability of Improvement (PI)**: just the probability of beating the current best; conservative (tends to over-exploit).

> 💡 **面试速查 / Interview cheat-sheet（★★★ AutoML/调参必考）**
> **中文**：**BO=GP 代理 + 采集函数**, 用最少评估优化**昂贵黑箱**(无梯度、有噪声、多局部最优)。循环:拟合 GP→最大化采集函数→评估→重复。**采集函数平衡探索(高σ)vs利用(高μ)**:**EI**(最常用, 期望改进)、**UCB**(μ+κσ, κ调探索)、**PI**(保守)。**为什么高效**:GP 的不确定性让它**避开已知差的区域、聚焦有希望的区域**, 比网格/随机搜索省一个数量级的评估。**主战场**:超参调优(AutoML)、实验设计、材料/药物/催化剂发现、机器人。**软肋**:①随维度升高变差(GP+采集在高维难,通常 <20 维);②GP 的 $O(n^3)$;③采集函数自身也要优化(非凸)。开源:Ax/BoTorch、Optuna、scikit-optimize、Spearmint。
> **English**: **BO = GP surrogate + acquisition function**, optimizing an **expensive black-box** (no gradients, noisy, multimodal) in the fewest evaluations. Loop: fit GP → maximize acquisition → evaluate → repeat. **Acquisition balances exploration (high σ) vs exploitation (high μ)**: **EI** (most common, expected improvement), **UCB** (μ+κσ, κ tunes exploration), **PI** (conservative). **Why efficient**: the GP's uncertainty lets it **avoid known-bad regions and focus on promising ones**, saving an order of magnitude of evaluations over grid/random search. **Home turf**: hyperparameter tuning (AutoML), experiment design, materials/drug/catalyst discovery, robotics. **Weaknesses**: ① degrades in high dimensions (GP + acquisition hard, usually <20-D); ② GP's $O(n^3)$; ③ the acquisition itself must be optimized (non-convex). Libraries: Ax/BoTorch, Optuna, scikit-optimize, Spearmint.


In [ ]:

# ============================================================
# 昂贵黑箱函数 + GP 代理 + EI 采集函数 / expensive black-box + GP surrogate + EI acquisition
# 中文:优化器【看不到】f 的公式, 只能花钱在某点评估一次拿到 f(x)。目标:用尽量少的评估找到最大值。
# English: the optimizer CANNOT see f's formula; it can only pay to evaluate f(x) at a point. Goal: find the max with few evals.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import norm
np.random.seed(0)
def black_box(x): return np.sin(3*x) + 0.5*np.sin(7*x) - 0.05*(x-1)**2   # 多峰"贵"函数 / multimodal expensive f
grid=np.linspace(-2,3,500)
true_max_x=grid[np.argmax(black_box(grid))]; true_max=black_box(true_max_x)

def rbf(A,B,l=0.35,sf=1.0):
    d=A[:,None]**2+B[None,:]**2-2*np.outer(A,B); return sf**2*np.exp(-0.5*d/l**2)
def gp_posterior(Xtr,ytr,Xs,sn=1e-4):                        # GP 代理(复用 18.6)/ GP surrogate
    K=rbf(Xtr,Xtr)+sn*np.eye(len(Xtr)); L=np.linalg.cholesky(K)
    a=np.linalg.solve(L.T,np.linalg.solve(L,ytr)); Ks=rbf(Xtr,Xs)
    mu=Ks.T@a; v=np.linalg.solve(L,Ks); sd=np.sqrt(np.maximum(np.diag(rbf(Xs,Xs))-np.sum(v**2,0),1e-9))
    return mu,sd
def expected_improvement(mu,sd,best,xi=0.01):               # EI 采集函数 / Expected Improvement
    imp=mu-best-xi; Z=imp/sd
    return imp*norm.cdf(Z)+sd*norm.pdf(Z)

print(f"真实最大值在 x={true_max_x:.2f}, f={true_max:.3f} (优化器不知道)/ true max (unknown to optimizer)")


**中文**：跑 **BO 循环**:从 3 个随机初始点开始,每轮拟合 GP、用 EI 选下一个点、评估、更新。我们记录每一轮的状态以便可视化"BO 是怎么一步步聪明地逼近最优的"。
**English**: Run the **BO loop**: start from 3 random points; each round fit the GP, pick the next point via EI, evaluate, update. We record each round's state to visualize "how BO cleverly homes in on the optimum step by step."


In [ ]:

# ============================================================
# BO 主循环 / the Bayesian optimization loop
# ============================================================
def run_bo(n_init=3, n_iter=12, acq="EI", kappa=2.0, seed=0):
    rng=np.random.default_rng(seed)
    X=list(rng.uniform(-2,3,n_init)); Y=[black_box(x) for x in X]   # 初始随机评估 / initial random evals
    snapshots=[]
    for t in range(n_iter):
        mu,sd=gp_posterior(np.array(X),np.array(Y),grid)           # 1) 拟合 GP 代理 / fit surrogate
        best=max(Y)
        acqv = expected_improvement(mu,sd,best) if acq=="EI" else mu+kappa*sd   # 2) 采集函数 / acquisition
        x_next=grid[np.argmax(acqv)]                               # 取采集函数最大点 / argmax acquisition
        snapshots.append((list(X),list(Y),mu.copy(),sd.copy(),acqv.copy(),x_next))
        X.append(x_next); Y.append(black_box(x_next))              # 3) 评估 + 更新 / evaluate & update
    return X, Y, snapshots

X, Y, snaps = run_bo(n_iter=12, acq="EI")
best_found=max(Y); best_x=X[int(np.argmax(Y))]
print(f"BO(EI) 15 次评估后找到: x={best_x:.2f}, f={best_found:.3f}")
print(f"距真实最优的差距 / gap to true max: {true_max-best_found:.4f} (≈0 = 找到了!)")


**中文**：可视化 **BO 的智能**。上排三张快照展示 BO 在不同轮次的状态:GP 代理(均值+不确定带)如何贴近真实黑箱、以及**采集函数(下方)如何在"预测高"和"不确定大"之间权衡、选出下一个评估点**。下排对比 BO(EI/UCB)和随机搜索的收敛速度。
**English**: Visualize **BO's intelligence**. The top row shows three snapshots at different rounds: how the GP surrogate (mean + uncertainty band) approaches the true black-box, and **how the acquisition function (below) trades off "high prediction" vs "high uncertainty" to pick the next point**. The bottom row compares BO (EI/UCB) vs random search in convergence speed.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig=plt.figure(figsize=(16,8))
# 上排:三个 BO 快照(代理+采集)/ top: three BO snapshots
for i,it in enumerate([1,5,11]):
    Xt,Yt,mu,sd,acqv,xn = snaps[it]
    ax=fig.add_subplot(2,3,i+1)
    ax.plot(grid, black_box(grid), "g--", alpha=0.6, label="真实黑箱 true (未知)")
    ax.plot(grid, mu, "b", label="GP 代理均值")
    ax.fill_between(grid, mu-2*sd, mu+2*sd, color="b", alpha=0.15)
    ax.scatter(Xt, Yt, c="k", zorder=5, label="已评估")
    ax.axvline(xn, color="r", ls=":", label="下一点(采集最大)")
    # 采集函数画在底部(缩放)/ acquisition at bottom (scaled)
    a2=acqv/ (acqv.max()+1e-9); ax.plot(grid, a2*1.0-2.3, "orange", alpha=0.8)
    ax.set_title(f"BO 第 {it+1} 轮 / iteration {it+1}"); ax.set_ylim(-2.5,2); ax.legend(fontsize=6,loc="upper right")
# 下排:BO vs 随机搜索 收敛 / bottom: convergence BO vs random
axc=fig.add_subplot(2,1,2)
def best_so_far_curve(method, seeds=25, n=15):
    curves=[]
    for s in range(seeds):
        if method=="random":
            r=np.random.default_rng(s); xs=r.uniform(-2,3,n); ys=black_box(xs)
            curves.append(np.maximum.accumulate(ys))
        else:
            Xr,Yr,_=run_bo(n_init=3,n_iter=n-3,acq=method,seed=s)
            curves.append(np.maximum.accumulate(Yr))
    return np.mean(curves,0)
for method,c,lab in [("EI","#4C72B0","BO-EI"),("UCB","#55A868","BO-UCB"),("random","#C44E52","随机搜索 random")]:
    axc.plot(range(1,16), best_so_far_curve(method), "o-", color=c, label=lab)
axc.axhline(true_max, ls="--", color="k", label="真实最优 true max")
axc.set_title("收敛:BO 用更少评估逼近最优 / BO reaches the optimum in fewer evaluations")
axc.set_xlabel("评估次数 #evaluations"); axc.set_ylabel("目前找到的最优 best-so-far"); axc.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/bay07_viz.png",dpi=80); plt.show()
print("BO 曲线(蓝/绿)远比随机(红)更快爬到真实最优线 —— 少评估、早收敛")


**中文**：诚实解读:
**English**: Honest takeaways:

**中文**：
1. **BO 用少得多的评估找到最优**:下排收敛图很直观——BO(EI/UCB)只用几次评估就爬到了真实最优线,而随机搜索慢吞吞、常卡在局部。在"每次评估都很贵"的场景(训一个大模型几小时、一次实验几千块),**省一个数量级的评估 = 省大量时间和金钱**。
2. **智能全在"用不确定性指导下一步"**:看上排快照的采集函数(橙线)——它**不会**只往当前 GP 均值最高的地方扎(那样会卡在局部最优),而是在"**预测高(利用)**"和"**不确定大(探索)**"之间权衡。早期不确定性大,BO 到处探索;后期聚焦到最有希望的峰精细逼近。这种自动的探索-利用平衡,是 BO 优于纯利用(爬山)和纯探索(随机)的根本。
3. **诚实的局限**:①**维度诅咒**——GP 和采集函数在高维(>约 20 维)都会失效,所以 BO 主要用于**中低维**问题(调十几个超参 OK,调神经网络的百万权重不行);②**继承 GP 的 $O(n^3)$**,评估次数上千就慢;③采集函数本身是**非凸**的,还得优化它(1 维可网格,高维要多起点/CMA-ES);④对**噪声大**的目标(如单次 A/B 的随机性)要用带噪 GP。所以 BO 的甜点是:**评估贵(所以值得花心思选点)、维度不太高、评估次数有限(几十到几百)**。

**English**:
1. **BO finds the optimum in far fewer evaluations**: the bottom convergence plot is telling — BO (EI/UCB) climbs to the true-optimum line in a handful of evaluations, while random search lags and often stalls locally. When each evaluation is expensive (hours to train a big model, thousands of dollars per experiment), **saving an order of magnitude of evaluations = huge time and money savings**.
2. **The intelligence is all in "using uncertainty to guide the next step"**: look at the acquisition (orange) in the top snapshots — it does **not** just rush to the current GP mean's peak (that would stall at a local optimum) but trades off "**high prediction (exploit)**" vs "**high uncertainty (explore)**." Early on, uncertainty is high and BO explores broadly; later it focuses and refines the most promising peak. This automatic explore-exploit balance is why BO beats pure exploitation (hill-climbing) and pure exploration (random).
3. **Honest limits**: ① the **curse of dimensionality** — both the GP and the acquisition fail in high dimensions (>~20-D), so BO is mainly for **low-to-medium-dim** problems (tuning a dozen hyperparameters is fine; the millions of NN weights are not); ② it **inherits the GP's $O(n^3)$**, slowing past a thousand evaluations; ③ the acquisition is itself **non-convex** and must be optimized (grid in 1-D, multi-start/CMA-ES in higher dims); ④ for **noisy** objectives (e.g. one A/B's randomness) use a noisy GP. So BO's sweet spot is: **expensive evaluations (worth thinking hard about where to sample), not-too-high dimensions, limited evaluation budget (tens to hundreds)**.

> 💼 **实战视角 / Practical angle**
> **中文**:BO 是**自动调参(AutoML)的事实标准**之一:①调机器学习超参(学习率、深度、正则——比网格/随机搜索省很多次训练);②**实验设计**(化学配方、材料、催化剂、药物分子——每次实验极贵);③A/B 与产品参数寻优;④机器人/控制的策略搜索。**工程**:用 **Optuna(默认 TPE, 也支持 GP)、Ax/BoTorch、scikit-optimize**;并行评估用批量 BO;类别/条件超参用对应核;噪声目标开 noisy 模式。**vs 随机搜索**:超参不多(<20)且训练贵时 BO 明显更省;维度极高或评估极便宜时随机搜索反而简单够用。面试金句:*"BO 用 GP 当昂贵黑箱的代理, 用采集函数(EI/UCB)平衡探索与利用来选下一个评估点, 少量评估就逼近最优; 适合贵、噪声、无梯度、中低维的优化——调参/实验设计的利器, 但高维会失效。"*
> **English**: BO is a **de facto standard for automated tuning (AutoML)**: ① tune ML hyperparameters (learning rate, depth, regularization — far fewer training runs than grid/random); ② **experiment design** (chemical formulations, materials, catalysts, drug molecules — each experiment very costly); ③ A/B and product-parameter optimization; ④ policy search for robotics/control. **Engineering**: use **Optuna (default TPE, also GP), Ax/BoTorch, scikit-optimize**; parallel evaluation via batch BO; categorical/conditional hyperparameters via appropriate kernels; enable noisy mode for noisy objectives. **vs random search**: with few hyperparameters (<20) and expensive training, BO is clearly cheaper; in very high dimensions or with very cheap evaluations, random search is simpler and often enough. Interview line: *"BO uses a GP surrogate of the expensive black-box and an acquisition function (EI/UCB) to balance exploration vs exploitation when choosing the next evaluation, reaching the optimum in few evaluations; ideal for expensive, noisy, gradient-free, low-to-medium-dim optimization — a powerhouse for tuning / experiment design, but it fails in high dimensions."*

---
### 小结 / Summary
- **中文**:BO=GP 代理 + 采集函数, 用最少评估优化昂贵黑箱; 循环:拟合GP→最大化采集→评估→重复。
- **English**: BO = GP surrogate + acquisition function, optimizing an expensive black-box in the fewest evaluations; loop: fit GP → maximize acquisition → evaluate → repeat.
- **中文**:采集函数(EI/UCB)平衡探索(高σ)与利用(高μ), 比随机/网格搜索省一个数量级评估。
- **English**: The acquisition (EI/UCB) balances exploration (high σ) and exploitation (high μ), saving an order of magnitude of evaluations over random/grid.
- **中文**:主战场=调参/实验设计(贵、噪声、无梯度、中低维); 高维与 $O(n^3)$ 是软肋。
- **English**: Home turf = tuning / experiment design (expensive, noisy, gradient-free, low-to-medium-dim); high dimensions and $O(n^3)$ are its weaknesses.

---
**中文**：🎉 至此 **Part 18 · 贝叶斯方法** 全部完成!从贝叶斯线性/逻辑回归 → MCMC → 变分推断 → 概率编程/层次模型 → 高斯过程 → 贝叶斯优化,你已经**从零实现**了贝叶斯机器学习的完整工具箱,建立了"后验=似然×先验""共轭 vs 近似(Laplace/MCMC/VI)""不确定性量化"的核心直觉——这正是**可信、可解释、数据高效**的机器学习之基石。
**English**: 🎉 **Part 18 · Bayesian Methods** is complete! From Bayesian linear/logistic regression → MCMC → variational inference → probabilistic programming/hierarchical models → Gaussian processes → Bayesian optimization, you have **implemented from scratch** the full toolbox of Bayesian ML, building core intuitions — posterior = likelihood × prior, conjugate vs approximate (Laplace/MCMC/VI), uncertainty quantification — the foundation of **trustworthy, interpretable, data-efficient** machine learning.
